In [2]:
#%pip install requests
#%pip install telethon
#%pip install --upgrade ipykernel
#%pip install requests lxml pandas
#%pip install selenium
#%pip install pymongo

In [3]:
#%pip install requests lxml mysql-connector-python

парсинг сайта с помощью scrapy


In [4]:
#%pip install crochet
#%pip install scrapy
import scrapy
from scrapy.crawler import CrawlerProcess
import crochet

In [5]:
from pathlib import Path

# БАЗА: папка, где лежит ноутбук (переносимо)
BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"                  # тут scrapy.cfg
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"   # тут пауки
SPIDER_FILE  = SPIDER_DIR / "oldgames_catalog.py"
OUTPUT_CSV   = BASE_PATH / "oldgames_dos.csv"

print("BASE_PATH   :", BASE_PATH.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SPIDER_DIR  :", SPIDER_DIR.resolve())
print("SPIDER_FILE :", SPIDER_FILE.resolve())
print("OUTPUT_CSV  :", OUTPUT_CSV.resolve())

# sanity-check
assert (PROJECT_ROOT / "scrapy.cfg").is_file(), "Не найден scrapy.cfg в папке проекта (simple_scrapy_spider)."

BASE_PATH   : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2
PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
SPIDER_DIR  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders
SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py
OUTPUT_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv


In [ ]:
SPIDER_DIR.mkdir(parents=True, exist_ok=True)

fixed_spider = r'''
import re
import scrapy

class OldGamesSpider(scrapy.Spider):
    name = "oldgames"
    allowed_domains = ["old-games.ru", "www.old-games.ru", "static.old-games.ru"]

    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.5,  # 
        "DEFAULT_REQUEST_HEADERS": {
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36 (Scrapy for research/edu)",
        },
        "FEED_EXPORT_ENCODING": "utf-8-sig",
    }

    def __init__(self, max_pages=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.page_counter = 0
        self.max_pages = int(max_pages) if max_pages else None
        self.start_urls = [
            "https://www.old-games.ru/catalog/?platform=1&sort=popularity"
        ]

    def parse(self, response):
        self.page_counter += 1

        # ВАЖНО: используем ТОЛЬКО прямые дети ./td[n] (первый столбец содержит вложенную таблицу)
        rows = response.xpath('//table[contains(@class,"listtable")]/tr[starts-with(@id,"game_")]')
        for row in rows:
            # 1) Название и ссылка
            name_link = row.xpath(
                './td[1]//a[starts-with(@href,"/game/") and '
                'not(contains(@href,"/video/")) and '
                'not(contains(@href,"/screenshots/")) and '
                'not(contains(@href,"/covers/"))][1]'
            )
            name = name_link.xpath('normalize-space(text())').get()
            url  = response.urljoin(name_link.xpath('@href').get())

            # 2) Жанр (может быть несколько)
            genres = row.xpath('./td[2]//a/text()').getall()
            genre = " / ".join([g.strip() for g in genres if g.strip()]) or None

            # 3) Год
            year_text = row.xpath('./td[3]//a/text()').re_first(r'\d{4}')
            year = int(year_text) if year_text else None

            # 4) Платформа (оставим fallback на текст)
            platforms = row.xpath('./td[4]//a/text()').getall()
            platform = " / ".join([p.strip() for p in platforms if p.strip()])
            if not platform:
                platform = row.xpath('normalize-space(./td[4])').get() or None

            # 5) Издатель 
            publishers = row.xpath('./td[5]//a/text()').getall()
            publisher = " / ".join([p.strip() for p in publishers if p.strip()])
            if not publisher:
                publisher = row.xpath('normalize-space(./td[5])').get() or None

            # 6) Оценка: число из title у <img> «… — N из 10»
            rating_title = row.xpath('./td[6]//img/@title').get()
            rating = None
            if rating_title:
                m = re.search(r'(\d+)\s*из\s*10', rating_title)
                if m:
                    rating = int(m.group(1))

            yield {
                "Название":  name or None,
                "Жанр":      genre or None,
                "Год":       year,
                "Платформа": platform or None,
                "Издатель":  publisher or None,
                "Оценка":    rating,
                "Ссылка": url,  
            }

        # Пагинация (rel="next" или кнопка ">")
        if self.max_pages is None or self.page_counter < self.max_pages:
            next_rel = response.xpath(
                '//ul[contains(@class,"pager")]//a[@rel="next"]/@href | '
                '//ul[contains(@class,"pager")]//a[normalize-space(text())=">"]/@href'
            ).get()
            if next_rel:
                yield response.follow(next_rel, callback=self.parse)
'''

SPIDER_FILE.write_text(fixed_spider.strip() + "\n", encoding="utf-8")
print("Паук обновлён:", SPIDER_FILE)


Паук обновлён: simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py


In [7]:
import sys, subprocess

cmd = [
    sys.executable, "-m", "scrapy", "crawl", "oldgames",
    "-O", str(OUTPUT_CSV.resolve()),  # абсолютный путь, собранный из относительного
    "-a", "max_pages=2",              # для теста: соберёт 2 страницы; если убрать параметр — пойдёт по всем
    "-s", "LOG_LEVEL=INFO",
]
print("Запуск:", " ".join(cmd))
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT.resolve()))
print("Exit code:", res.returncode)
print("CSV:", OUTPUT_CSV.resolve(), "=>", OUTPUT_CSV.exists())

Запуск: c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Scripts\python.exe -m scrapy crawl oldgames -O C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv -a max_pages=2 -s LOG_LEVEL=INFO
Exit code: 0
CSV: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv => True


In [8]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
display(df.head(20))

print("\nПроверки на пустые значения:")
for col in ["Жанр", "Платформа", "Издатель"]:
    print(f"{col}: пустых = {int(df[col].isna().sum())}")

,Название,Жанр,Год,Платформа,Издатель,Оценка,Ссылка
0,WarCraft II: Tides of Darkness,Strategy,1995,DOS,Blizzard Entertainment,10,https://www.old-games.ru/game/73.html
1,Blood,Action,1997,DOS,GT Interactive Software,10,https://www.old-games.ru/game/11.html
2,Quake,Action,1996,DOS,id Software,10,https://www.old-games.ru/game/64.html
3,X-COM: UFO Defense,Strategy,1994,DOS,MicroProse Software,8,https://www.old-games.ru/game/77.html
4,DOOM,Action,1993,DOS,id Software,10,https://www.old-games.ru/game/4788.html
5,Dune II: The Building of a Dynasty,Strategy,1992,DOS,Virgin Games,10,https://www.old-games.ru/game/1482.html
6,Wolfenstein 3D,Action,1992,DOS,Apogee Software,8,https://www.old-games.ru/game/76.html
7,Sid Meier's Civilization,Strategy,1991,DOS,MicroProse Software,10,https://www.old-games.ru/game/12.html
8,Duke Nukem 3D: Atomic Edition,Action,1996,DOS,GT Interactive Software,10,https://www.old-games.ru/game/604.html
9,Prince of Persia,Arcade,1990,DOS,Brøderbund Software,10,https://www.old-games.ru/game/62.html



Проверки на пустые значения:
Жанр: пустых = 0
Платформа: пустых = 0
Издатель: пустых = 0


In [1]:
from pathlib import Path

BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"

SPIDER_FILE  = SPIDER_DIR / "olx_transport.py"

MASTER_CSV   = BASE_PATH / "olx_transport_master.csv"   # “накопительный” файл
RUN_CSV      = BASE_PATH / "olx_transport_run.csv"      # файл текущего запуска (временный)

SPIDER_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SPIDER_FILE :", SPIDER_FILE.resolve())
print("MASTER_CSV  :", MASTER_CSV.resolve())
print("RUN_CSV     :", RUN_CSV.resolve())

assert (PROJECT_ROOT / "scrapy.cfg").is_file(), "Не найден scrapy.cfg в папке проекта (simple_scrapy_spider)."


PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\olx_transport.py
MASTER_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_master.csv
RUN_CSV     : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv


In [ ]:
fixed_spider = r'''
import re
import scrapy


def _norm_spaces(s: str | None) -> str | None:
    if not s:
        return None
    return re.sub(r"\s+", " ", s).strip()


def _re_first(text: str, pattern: str) -> str | None:
    m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
    return m.group(1).strip() if m else None


def _kv(text: str, labels: list[str]) -> str | None:
    # Ищем "Лейбл: значение" в тексте страницы
    for lbl in labels:
        m = re.search(rf"^{re.escape(lbl)}\s*:\s*(.+)$", text, flags=re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1).strip()
    return None


def _guess_model_from_title(title: str | None) -> str | None:
    # очень грубый fallback: убираем год (4 цифры) и хвост после него
    if not title:
        return None
    t = _norm_spaces(title)
    m = re.search(r"^(.*?)(?:\b(19|20)\d{2}\b.*)?$", t)
    base = m.group(1).strip() if m else t
    return base or t


class OlxTransportSpider(scrapy.Spider):
    name = "olx_transport"
    allowed_domains = ["olx.ua", "www.olx.ua"]

    custom_settings = {
        # Важно: если ROBOTSTXT_OBEY=True и robots запрещает — паук может собрать 0 строк.
        
        "ROBOTSTXT_OBEY": False,

        "DOWNLOAD_DELAY": 0.7,
        "AUTOTHROTTLE_ENABLED": True,
        "FEED_EXPORT_ENCODING": "utf-8-sig",
        "DEFAULT_REQUEST_HEADERS": {
            "Accept-Language": "uk-UA,uk;q=0.9,ru-RU;q=0.8,ru;q=0.7,en-US;q=0.6,en;q=0.5",
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
        },
    }

    def __init__(self, start_url=None, max_pages=1, max_items=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_urls = [start_url or "https://www.olx.ua/uk/transport/"]
        self.max_pages = int(max_pages) if max_pages else None
        self.max_items = int(max_items) if max_items else None
        self.page_counter = 0
        self.item_counter = 0

    def parse(self, response):
        self.page_counter += 1

        cards = response.xpath('//div[@data-cy="l-card"]')
        for card in cards:
            if self.max_items and self.item_counter >= self.max_items:
                break

            title = _norm_spaces(card.xpath('.//*[@data-testid="ad-card-title"]/text()').get())
            price = _norm_spaces(card.xpath('.//*[@data-testid="ad-price"]/text()').get())

            href = card.xpath('.//a[contains(@href,"/d/")]/@href').get()
            if not href:
                continue
            url = response.urljoin(href.split("?")[0])

            # Короткие характеристики часто “спрятаны” в span с data-testid вида "*card-param*"
            short_params = card.xpath('.//span[contains(@data-testid,"card-param")]/text()').getall()
            short_params = ", ".join([_norm_spaces(x) for x in short_params if _norm_spaces(x)]) or None

            self.item_counter += 1
            yield response.follow(
                url,
                callback=self.parse_offer,
                meta={
                    "listing_title": title,
                    "listing_price": price,
                    "listing_short": short_params,
                    "listing_url": url,
                },
            )

        # пагинация (на OLX есть кнопка вперёд)
        if self.max_pages is None or self.page_counter < self.max_pages:
            next_href = response.xpath('//a[@data-testid="pagination-forward"]/@href').get()
            if next_href:
                yield response.follow(next_href, callback=self.parse)

    def parse_offer(self, response):
        # Берём много “человеческого” текста страницы, чтобы доставать "Лейбл: Значение"
        texts = [t.strip() for t in response.xpath('//text()').getall() if t.strip()]
        joined = "\n".join(texts)

        # Название
        title = _norm_spaces(response.xpath('normalize-space(//h1[1])').get()) or response.meta.get("listing_title")

        # ID (на OLX обычно есть строка вида "ID: 123456789")
        ad_id = _re_first(joined, r"\bID:\s*(\d+)\b")

        # Цена (fallback на то, что было на карточке)
        price = _re_first(joined, r"^###\s*([0-9\s]+(?:грн\.|UAH|₴).*)$")  # иногда в тексте встречается как заголовок
        price = _norm_spaces(price) or response.meta.get("listing_price")

        # Модель: пытаемся собрать "Марка + Модель" из параметров
        brand = _kv(joined, ["Марка", "Бренд"])
        model_only = _kv(joined, ["Модель"])
        model = " ".join([x for x in [brand, model_only] if x]) or _guess_model_from_title(title)

        # Состояние: "Стан" (укр) или "Состояние" (рус)
        condition = _kv(joined, ["Стан", "Состояние", "Стан авто"])

        # Краткие характеристики: собираем несколько “типовых” полей
        char_labels = [
            "Рік випуску", "Год выпуска",
            "Пробіг", "Пробег",
            "Тип кузова", "Тип палива", "Вид палива",
            "Коробка передач",
            "Об'єм двигуна", "Объем двигателя",
            "Кількість дверей", "Количество дверей",
            "Колір", "Цвет",
        ]
        parts = []
        for lbl in char_labels:
            v = _kv(joined, [lbl])
            if v:
                parts.append(f"{lbl}: {v}")
        characteristics = "; ".join(parts) if parts else response.meta.get("listing_short")

        yield {
            "ID": ad_id,
            "Название": title,
            "Модель": model,
            "Краткие характеристики": characteristics,
            "Состояние": condition,
            "Цена": price,
            "Ссылка": response.meta.get("listing_url") or response.url,
        }
'''

SPIDER_FILE.write_text(fixed_spider.strip() + "\n", encoding="utf-8")
print("Паук сохранён:", SPIDER_FILE)


Паук сохранён: simple_scrapy_spider\simple_scrapy_spider\spiders\olx_transport.py


In [3]:
import sys, subprocess

cmd = [
    sys.executable, "-m", "scrapy", "crawl", "olx_transport",
    "-O", str(RUN_CSV.resolve()),      # перезаписываем ТОЛЬКО временный файл запуска
    "-a", "max_pages=1",             
    # "-a", "start_url=https://www.olx.ua/uk/transport/legkovye-avtomobili/",  
    "-s", "LOG_LEVEL=INFO",
]

print("Запуск:", " ".join(cmd))
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT.resolve()))
print("Exit code:", res.returncode)
print("RUN_CSV exists:", RUN_CSV.exists(), "=>", RUN_CSV.resolve())


Запуск: c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Scripts\python.exe -m scrapy crawl olx_transport -O C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv -a max_pages=1 -s LOG_LEVEL=INFO
Exit code: 0
RUN_CSV exists: True => C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_run.csv


In [5]:
import pandas as pd

new_df = pd.read_csv(RUN_CSV, encoding="utf-8-sig")

if MASTER_CSV.exists():
    old_df = pd.read_csv(MASTER_CSV, encoding="utf-8-sig")
    df = pd.concat([old_df, new_df], ignore_index=True)
else:
    df = new_df.copy()

# Дедупликация: сначала по ID, если пусто — по ссылке
key = df["ID"].fillna(df["Ссылка"]) if "ID" in df.columns else df["Ссылка"]
df["__key"] = key

df = df.drop_duplicates(subset="__key", keep="first").drop(columns="__key")

df.to_csv(MASTER_CSV, index=False, encoding="utf-8-sig")

display(df.head(30))
print("Итоговых строк:", len(df))
print("MASTER_CSV:", MASTER_CSV.resolve())


,ID,Название,Модель,Краткие характеристики,Состояние,Цена,Ссылка
0,870576653,NaN,NaN,Рік випуску: 2007,З пробігом,358 632 грн.,https://www.olx.ua/d/uk/obyavlenie/konteynerov...
1,908021831,NaN,NaN,Рік випуску: 2025; Пробіг: 0 км,Новий,75 000 грн.,https://www.olx.ua/d/uk/obyavlenie/pritsep-1pt...
2,908882237,NaN,Octavia,Рік випуску: 2007; Пробіг: 300 тис.км.; Тип ку...,NaN,88 603 грн.,https://www.olx.ua/d/uk/obyavlenie/skoda-oktav...
3,909502836,NaN,Corsa,Рік випуску: 2008; Пробіг: 147 тис.км.; Тип ку...,NaN,208 850 грн.,https://www.olx.ua/d/uk/obyavlenie/opel-corsa-...
4,908087619,NaN,100,Рік випуску: 1991; Пробіг: 350 тис.км.; Тип ку...,NaN,118 137 грн.,https://www.olx.ua/d/uk/obyavlenie/prodam-aud-...
5,837119994,NaN,NaN,Рік випуску: 2025; Пробіг: 5 км; Коробка перед...,Новий,63 000 грн.,https://www.olx.ua/d/uk/obyavlenie/vagi-na-fro...
6,909790254,NaN,Sportage,Рік випуску: 2006; Пробіг: 329 тис.км.; Тип ку...,NaN,202 521 грн.,https://www.olx.ua/d/uk/obyavlenie/ka-sportege...
7,906404557,NaN,NaN,Рік випуску: 2025,Новий,242 000 грн.,https://www.olx.ua/d/uk/obyavlenie/elvorti-ast...
8,909185071,NaN,NaN,Рік випуску: 1995; Пробіг: 2 км,З пробігом,274 205 грн.,https://www.olx.ua/d/uk/obyavlenie/kemper-kara...
9,908601489,NaN,5 серія,Рік випуску: 2001; Пробіг: 300 тис.км.; Тип ку...,NaN,181 425 грн.,https://www.olx.ua/d/uk/obyavlenie/prodam-bmw-...


Итоговых строк: 52
MASTER_CSV: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\olx_transport_master.csv


попробуем распарсить сайт с новостями спорта

In [36]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Tuple

def _clean_text(s: Optional[str]) -> str:
    if not s:
        return ""
    s = s.replace("\u2009", " ").replace("\xa0", " ")
    return re.sub(r"\s+", " ", s).strip()

@dataclass
class ParseDiag:
    ok: bool
    kind: str
    url_requested: str
    url_final: Optional[str] = None
    status_code: Optional[int] = None
    message: str = ""
    evidence: Dict[str, Any] = None

def log_line(stage: str, msg: str):
    print(f"[{stage}] {msg}")

def parse_news_bs4(html: str, base_url: str = "https://football24.ru/") -> List[Dict[str, Any]]:
    soup = BeautifulSoup(html, "lxml")
    items = []

    for art in soup.select("#dle-content article"):
        a = art.select_one("h2[itemprop='headline'] a") or art.select_one("h2 a")
        t = art.select_one("time[itemprop='datePublished']") or art.select_one("time")
        p = art.select_one("p[itemprop='description']") or art.select_one("p")

        title = _clean_text(a.get_text()) if a else ""
        url = (a.get("href") or "").strip() if a else ""
        if url and url.startswith("/"):
            url = base_url.rstrip("/") + url
        elif url and not url.startswith("http"):
            url = base_url.rstrip("/") + "/" + url.lstrip("/")

        date_iso = _clean_text(t.get("datetime", "")) if t else ""
        date_txt = _clean_text(t.get_text()) if t else ""
        date_val = date_iso if date_iso else date_txt

        text = _clean_text(p.get_text()) if p else ""

        if title and url:
            items.append({"title": title, "date": date_val, "text": text, "url": url})

    return items

def looks_like_real_check_page(html: str) -> bool:
    
    low = (html or "").lower()
    strong = [
        "verify you are human", "attention required", "/cdn-cgi/", "cf-ray",
        "g-recaptcha", "h-captcha", "подтвердите что вы не робот", "проверка безопасности"
    ]
    return any(m in low for m in strong)

def bs4_collect_news(max_items: int = 80, max_pages: int = 20, timeout: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame]:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.7,en;q=0.6",
    })

    items_all: List[Dict[str, Any]] = []
    errors: List[Dict[str, Any]] = []
    seen = set()

    for page in range(1, max_pages + 1):
        url = "https://football24.ru/allnews/" if page == 1 else f"https://football24.ru/allnews/page/{page}/"
        log_line("BS4", f"GET page={page} url={url}")

        try:
            r = session.get(url, timeout=timeout, allow_redirects=True)
            html = r.text or ""

            news = parse_news_bs4(html, base_url="https://football24.ru/")

            # ЛОГ: что нашли
            log_line("BS4", f"status={r.status_code} found={len(news)} total={len(items_all)}")

            # если новостей 0 — диагностика и стоп
            if len(news) == 0:
                # сохраняем html для ручной проверки
                try:
                    with open(f"football24_bs4_debug_page{page}.html", "w", encoding="utf-8") as f:
                        f.write(html)
                except:
                    pass

                if looks_like_real_check_page(html):
                    errors.append(asdict(ParseDiag(
                        ok=False, kind="CHECK_PAGE", url_requested=url, url_final=str(r.url),
                        status_code=r.status_code,
                        message="HTTP 200, но похоже на страницу проверки/антибота. Сохранён football24_bs4_debug_pageN.html",
                        evidence={}
                    )))
                    break

                errors.append(asdict(ParseDiag(
                    ok=False, kind="NO_ARTICLES", url_requested=url, url_final=str(r.url),
                    status_code=r.status_code,
                    message="Не найдено #dle-content article. Возможно другая верстка/шаблон. Сохранён football24_bs4_debug_pageN.html",
                    evidence={}
                )))
                break

            # добавляем уникальные по url
            added = 0
            for it in news:
                if it["url"] in seen:
                    continue
                seen.add(it["url"])
                items_all.append(it)
                added += 1
                if len(items_all) >= max_items:
                    break

            log_line("BS4", f"added={added} new_total={len(items_all)}")

            if len(items_all) >= max_items:
                break

        except requests.Timeout as e:
            errors.append(asdict(ParseDiag(False, "TIMEOUT", url, None, None, str(e), {})))
            log_line("BS4", f"TIMEOUT | {e}")
            break
        except requests.RequestException as e:
            errors.append(asdict(ParseDiag(False, "CONNECTION_ERROR", url, None, None, str(e), {})))
            log_line("BS4", f"CONNECTION_ERROR | {e}")
            break
        except Exception as e:
            errors.append(asdict(ParseDiag(False, "UNKNOWN", url, None, None, str(e), {})))
            log_line("BS4", f"UNKNOWN | {e}")
            break

    return pd.DataFrame(items_all), pd.DataFrame(errors)

df_bs4, err_bs4 = bs4_collect_news(max_items=200, max_pages=20)
print("RESULT:", len(df_bs4), "news;", len(err_bs4), "errors")
display(df_bs4.head(20))
if len(err_bs4):
    display(err_bs4.head(10))


[BS4] GET page=1 url=https://football24.ru/allnews/
[BS4] status=200 found=20 total=0
[BS4] added=20 new_total=20
[BS4] GET page=2 url=https://football24.ru/allnews/page/2/
[BS4] status=200 found=20 total=20
[BS4] added=20 new_total=40
[BS4] GET page=3 url=https://football24.ru/allnews/page/3/
[BS4] status=200 found=20 total=40
[BS4] added=20 new_total=60
[BS4] GET page=4 url=https://football24.ru/allnews/page/4/
[BS4] status=200 found=20 total=60
[BS4] added=20 new_total=80
[BS4] GET page=5 url=https://football24.ru/allnews/page/5/
[BS4] status=200 found=20 total=80
[BS4] added=20 new_total=100
[BS4] GET page=6 url=https://football24.ru/allnews/page/6/
[BS4] status=200 found=20 total=100
[BS4] added=20 new_total=120
[BS4] GET page=7 url=https://football24.ru/allnews/page/7/
[BS4] status=200 found=20 total=120
[BS4] added=20 new_total=140
[BS4] GET page=8 url=https://football24.ru/allnews/page/8/
[BS4] status=200 found=20 total=140
[BS4] added=20 new_total=160
[BS4] GET page=9 url=http

,title,date,text,url
0,Месси назвал свой самый любимый гол в карьере,2025-12-25T18:03,Мяч был забит 14 лет назад.,https://football24.ru/allnews/272588-messi-naz...
1,Бышовец: «Зенит» ни в чем не превосходит «Крас...,2025-12-25T17:35,Клубы ведут борьбу за чемпионство.,https://football24.ru/allnews/russia/rpl-premi...
2,«Челси» интересуется бывшим нападающим «Барсел...,2025-12-25T17:21,Бразилец может перебраться в АПЛ.,https://football24.ru/allnews/england/apl-prem...
3,«Барселона» приняла решение по будущему Кристе...,2025-12-25T17:00,Датчанин выступает за сине-гранатовых с 2022 г...,https://football24.ru/allnews/spain/la-liga/27...
4,«Ливерпуль» может перехватить трансферную цель...,2025-12-25T16:45,Гонка за игроком началась.,https://football24.ru/allnews/england/apl-prem...
5,«Зенит» готов заплатить 50 млн евро за 19-летн...,2025-12-25T16:29,Может состояться рекордный трансфер.,https://football24.ru/allnews/russia/rpl-premi...
6,Баркола выдвинул условие «Ливерпулю»,2025-12-25T16:18,Красные хотят подписать игрока.,https://football24.ru/allnews/england/apl-prem...
7,Дмитрий Баринов назвал лучшего игрока первой п...,2025-12-25T16:11,Выделил партнера по сборной.,https://football24.ru/allnews/russia/rpl-premi...
8,ЦСКА собирается отправить своего нападающего в...,2025-12-25T15:45,Заинтересованы четыре клуба.,https://football24.ru/allnews/russia/rpl-premi...
9,Определены главные фавориты на «Золотой мяч» в...,2025-12-25T15:27,Борьбу ведут пять лидеров своих клубов.,https://football24.ru/allnews/spain/la-liga/27...


Создаем файл спайдера пока без ухищрений

In [ ]:
#определяем пути, директории
from pathlib import Path

BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"

# Основной паук (который мы хотим запускать)
SPIDER_FILE  = SPIDER_DIR / "football24_allnews.py"

# Файлы данных
MASTER_CSV   = BASE_PATH / "football24_master.csv"     # накопительный файл
RUN_CSV      = BASE_PATH / "football24_run.csv"        # файл текущего запуска (временный)
ERRORS_JSON  = BASE_PATH / "football24_errors.json"    # диагностика/ошибки (json)

SPIDER_DIR.mkdir(parents=True, exist_ok=True)

print("📁 PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("🕷️  SPIDER_DIR  :", SPIDER_DIR.resolve())
print("📝 SPIDER_FILE :", SPIDER_FILE.resolve())
print("📦 MASTER_CSV  :", MASTER_CSV.resolve())
print("🧪 RUN_CSV     :", RUN_CSV.resolve())
print("⚠️  ERRORS_JSON :", ERRORS_JSON.resolve())

assert (PROJECT_ROOT / "scrapy.cfg").is_file(), (
    "❌ Не найден scrapy.cfg в папке проекта (simple_scrapy_spider). "
    "нужно проверить, что PROJECT_ROOT указывает на корень scrapy-проекта."
)




📁 PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
🕷️  SPIDER_DIR  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders
📝 SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\football24_allnews.py
📦 MASTER_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\football24_master.csv
🧪 RUN_CSV     : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\football24_run.csv
⚠️  ERRORS_JSON : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\football24_errors.json


In [ ]:
spider_code = r'''
import json
import re
from pathlib import Path
import scrapy
from scrapy.exceptions import CloseSpider


def clean(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\u00a0", " ").replace("\u2009", " ")
    return re.sub(r"\s+", " ", s).strip()


class Football24AllNewsSpider(scrapy.Spider):
    name = "football24_allnews"
    allowed_domains = ["football24.ru"]

    custom_settings = {
        "LOG_LEVEL": "ERROR",            # будем логировать сами
        "TELNETCONSOLE_ENABLED": False,
        "ROBOTSTXT_OBEY": False,         # иначе robots может мешать /page/
        "DOWNLOAD_TIMEOUT": 25,
        "RETRY_ENABLED": True,
        "RETRY_TIMES": 2,
        "CONCURRENT_REQUESTS": 2,
        "DOWNLOAD_DELAY": 0.2,
        "AUTOTHROTTLE_ENABLED": True,
        "AUTOTHROTTLE_START_DELAY": 0.5,
        "AUTOTHROTTLE_MAX_DELAY": 6.0,
        "FEED_EXPORT_ENCODING": "utf-8-sig",
        "USER_AGENT": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "DEFAULT_REQUEST_HEADERS": {
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.7,en;q=0.6",
        },
    }

    def __init__(self, max_items=80, max_pages=20, errors_json="football24_errors.json", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_items = int(max_items)
        self.max_pages = int(max_pages)
        self.errors_json = str(errors_json)

        self.total = 0
        self.page = 0
        self.seen = set()
        self.errors = []

    # ---------- удобный лог ----------
    def log_human(self, msg: str):
        print(f"[PAUK] {msg}")

    # ---------- ошибки ----------
    def push_error(self, kind: str, response, message: str, evidence=None):
        rec = {
            "ok": False,
            "kind": kind,
            "url_requested": getattr(getattr(response, "request", None), "url", None),
            "url_final": getattr(response, "url", None),
            "status_code": getattr(response, "status", None),
            "message": message,
            "evidence": evidence or {},
        }
        self.errors.append(rec)
        try:
            Path(self.errors_json).write_text(json.dumps(self.errors, ensure_ascii=False, indent=2), encoding="utf-8")
        except Exception:
            pass

    def save_debug(self, response, page: int):
        
        try:
            Path(f"football24_debug_page{page}.html").write_text(response.text or "", encoding="utf-8", errors="ignore")
        except Exception:
            pass

    
    # "блок" определяем ТОЛЬКО если новостей (article) не найдено + есть сильные маркеры.
    def is_real_check_page(self, response) -> bool:
        html = (response.text or "").lower()

        strong_markers = [
            "g-recaptcha", "h-captcha", "cf-ray", "/cdn-cgi/", "attention required",
            "verify you are human", "подтвердите что вы не робот", "проверка безопасности",
        ]
        return any(m in html for m in strong_markers)

    # ---------- старт: только первая страница, дальше идём последовательно ----------
    def start_requests(self):
        yield scrapy.Request("https://football24.ru/allnews/", callback=self.parse_page, meta={"page": 1}, dont_filter=True)

    def parse_page(self, response):
        page = int(response.meta.get("page", 1))
        self.page = page

        # 1) СНАЧАЛА парсим новости
        articles = response.css("#dle-content article")
        found = len(articles)

        if found > 0:
            self.log_human(f"Страница {page}: найдено {found} новостей (HTTP {response.status})")

            for art in articles:
                href = art.css("h2[itemprop='headline'] a::attr(href)").get() or art.css("h2 a::attr(href)").get()
                title = art.css("h2[itemprop='headline'] a::text").get() or art.css("h2 a::text").get()
                dt_iso = art.css("time[itemprop='datePublished']::attr(datetime)").get() or art.css("time::attr(datetime)").get()
                dt_txt = art.css("time[itemprop='datePublished']::text").get() or art.css("time::text").get()
                desc = art.css("p[itemprop='description']::text").get() or art.css("p::text").get()

                url = response.urljoin((href or "").strip())
                title = clean(title)
                date_val = clean(dt_iso) if dt_iso else clean(dt_txt)
                text = clean(desc)

                if not url or not title:
                    continue
                if url in self.seen:
                    continue

                self.seen.add(url)
                self.total += 1

                yield {"title": title, "date": date_val, "text": text, "url": url}

                if self.total >= self.max_items:
                    self.log_human(f"Готово: собрано {self.total} новостей (лимит max_items).")
                    raise CloseSpider("MAX_ITEMS")

            self.log_human(f"Итого после стр. {page}: всего {self.total}")

        else:
            # 2) Если 0 новостей — сохраняем HTML и диагностируем
            self.save_debug(response, page)
            if self.is_real_check_page(response):
                self.push_error("CHECK_PAGE", response, "Получена страница проверки/антибота. Сохранён football24_debug_pageN.html")
                self.log_human(f"Останов: страница проверки (см. football24_debug_page{page}.html)")
                raise CloseSpider("CHECK_PAGE")

            # Не проверка — значит верстка изменилась или пришёл другой шаблон
            self.push_error("NO_ARTICLES", response, "Не найдено #dle-content article. Сохранён football24_debug_pageN.html")
            self.log_human(f"Останов: нет article (см. football24_debug_page{page}.html)")
            raise CloseSpider("NO_ARTICLES")

        # 3) Следующая страница (последовательно, чтобы STOP реально останавливал)
        if page >= self.max_pages:
            self.log_human(f"Останов: достигнут лимит страниц {self.max_pages}.")
            raise CloseSpider("MAX_PAGES")

        next_page = page + 1
        next_url = f"https://football24.ru/allnews/page/{next_page}/"
        yield scrapy.Request(next_url, callback=self.parse_page, meta={"page": next_page}, dont_filter=True)

    def closed(self, reason):
        # всегда сохраняем ошибки (даже если 0)
        try:
            Path(self.errors_json).write_text(json.dumps(self.errors, ensure_ascii=False, indent=2), encoding="utf-8")
        except Exception:
            pass
        self.log_human(f"Завершено: причина={reason} | всего={self.total} | ошибок={len(self.errors)}")
'''

SPIDER_FILE.write_text(spider_code.strip() + "\n", encoding="utf-8")
print("OK: spider обновлён:", SPIDER_FILE.resolve())


OK: spider обновлён: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\football24_allnews.py


In [39]:
import os, sys, subprocess, json
import pandas as pd
from IPython.display import display

os.environ["PYTHONIOENCODING"] = "utf-8"

def run_and_report(max_items=80, max_pages=20):
    # почистим файлы прогона
    for f in [RUN_CSV, ERRORS_JSON]:
        try:
            if f.exists():
                f.unlink()
        except Exception:
            pass

    cmd = [
        sys.executable, "-m", "scrapy", "crawl", "football24_allnews",
        "-O", str(RUN_CSV.resolve()),
        "-a", f"max_items={max_items}",
        "-a", f"max_pages={max_pages}",
        "-a", f"errors_json={ERRORS_JSON.resolve()}",
        "-s", "LOG_LEVEL=ERROR",
    ]

    res = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT.resolve()),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=os.environ.copy(),
    )

    # берём только наш лог
    custom_log = "\n".join([ln for ln in (res.stdout or "").splitlines() if ln.startswith("[PAUK]")])

    # читаем результаты
    df = pd.read_csv(RUN_CSV, encoding="utf-8-sig") if RUN_CSV.exists() and RUN_CSV.stat().st_size else pd.DataFrame()
    errs = json.loads(ERRORS_JSON.read_text(encoding="utf-8")) if ERRORS_JSON.exists() and ERRORS_JSON.stat().st_size else []
    df_err = pd.DataFrame(errs)

    print("\n" + "═"*56)
    print("ОТЧЁТ: Football24 новости (Scrapy spider)")
    print("═"*56)
    print(f"Код завершения процесса: {res.returncode}")
    print(f"Собрано новостей: {len(df)}")
    print(f"Записано диагностик: {len(df_err)}")

    if custom_log.strip():
        print("\nЛОГ (коротко):")
        print(custom_log)
    else:
        print("\nЛОГ: (пусто)")

    if not df.empty:
        print("\nПервые 20 новостей:")
        display(df.head(20))

    if not df_err.empty:
        print("\nДиагностика (первые 5):")
        display(df_err.head(5))
        last = df_err.iloc[-1].to_dict()
        print("\nПоследняя причина:")
        print(f"- kind  : {last.get('kind')}")
        print(f"- status: {last.get('status_code')}")
        print(f"- msg   : {last.get('message')}")
        print(f"- url   : {last.get('url_final')}")

    print("═"*56 + "\n")

run_and_report(max_items=80, max_pages=20)



════════════════════════════════════════════════════════
ОТЧЁТ: Football24 новости (Scrapy spider)
════════════════════════════════════════════════════════
Код завершения процесса: 0
Собрано новостей: 80
Записано диагностик: 0

ЛОГ (коротко):
[PAUK] Страница 1: найдено 20 новостей (HTTP 200)
[PAUK] Итого после стр. 1: всего 20
[PAUK] Страница 2: найдено 20 новостей (HTTP 200)
[PAUK] Итого после стр. 2: всего 40
[PAUK] Страница 3: найдено 20 новостей (HTTP 200)
[PAUK] Итого после стр. 3: всего 60
[PAUK] Страница 4: найдено 20 новостей (HTTP 200)
[PAUK] Готово: собрано 80 новостей (лимит max_items).
[PAUK] Завершено: причина=MAX_ITEMS | всего=80 | ошибок=0

Первые 20 новостей:


,title,date,text,url
0,Месси назвал свой самый любимый гол в карьере,2025-12-25T18:03,Мяч был забит 14 лет назад.,https://football24.ru/allnews/272588-messi-naz...
1,Бышовец: «Зенит» ни в чем не превосходит «Крас...,2025-12-25T17:35,Клубы ведут борьбу за чемпионство.,https://football24.ru/allnews/russia/rpl-premi...
2,«Челси» интересуется бывшим нападающим «Барсел...,2025-12-25T17:21,Бразилец может перебраться в АПЛ.,https://football24.ru/allnews/england/apl-prem...
3,«Барселона» приняла решение по будущему Кристе...,2025-12-25T17:00,Датчанин выступает за сине-гранатовых с 2022 г...,https://football24.ru/allnews/spain/la-liga/27...
4,«Ливерпуль» может перехватить трансферную цель...,2025-12-25T16:45,Гонка за игроком началась.,https://football24.ru/allnews/england/apl-prem...
5,«Зенит» готов заплатить 50 млн евро за 19-летн...,2025-12-25T16:29,Может состояться рекордный трансфер.,https://football24.ru/allnews/russia/rpl-premi...
6,Баркола выдвинул условие «Ливерпулю»,2025-12-25T16:18,Красные хотят подписать игрока.,https://football24.ru/allnews/england/apl-prem...
7,Дмитрий Баринов назвал лучшего игрока первой п...,2025-12-25T16:11,Выделил партнера по сборной.,https://football24.ru/allnews/russia/rpl-premi...
8,ЦСКА собирается отправить своего нападающего в...,2025-12-25T15:45,Заинтересованы четыре клуба.,https://football24.ru/allnews/russia/rpl-premi...
9,Определены главные фавориты на «Золотой мяч» в...,2025-12-25T15:27,Борьбу ведут пять лидеров своих клубов.,https://football24.ru/allnews/spain/la-liga/27...


════════════════════════════════════════════════════════

